In [ ]:
import os

ROOT_DIR = os.getcwd()
EPICS_FOLDER = os.path.join(ROOT_DIR, "Epics")
OUTPUT_FOLDER = os.path.join(ROOT_DIR, "preprocessed data")

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# 🔥 SELECT FILE HERE
FILE_NAME = "vinod_2nd_rglr.pdb" 


'''Completed = kanpur-1st-c3.pdb, kanput-1st-c2.pdb, kanpur_1st_c4.pdb, moib_complex_2nd.pdb,
    moib_complex1st.pdb, moib_complex-3rd.pdb, '''
''' 
     
     
     
    
    vinod_2nd_rglr.pdb
'''

FILE_PATH = os.path.join(EPICS_FOLDER, FILE_NAME)

OUTPUT_FILE = os.path.join(
    OUTPUT_FOLDER,
    FILE_NAME.replace(".pdb", "_normalized.hdf5")
)

print("📂 Input:", FILE_PATH)
print("💾 Output:", OUTPUT_FILE)

In [ ]:
import numpy as np

def read_pdb_fixed_frames(file_path, atoms_per_frame=10000):
    coords = []

    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith(("ATOM", "HETATM")):
                try:
                    x = float(line[30:38])
                    y = float(line[38:46])
                    z = float(line[46:54])
                    coords.append([x, y, z])
                except:
                    continue

    coords = np.array(coords, dtype=np.float32)

    total_atoms = len(coords)

    n_frames = total_atoms // atoms_per_frame

    trimmed = coords[:n_frames * atoms_per_frame]

    frames = trimmed.reshape(n_frames, atoms_per_frame, 3)

    print(f"✅ Extracted {n_frames} frames")
    print(f"📏 Each frame: {atoms_per_frame} atoms")

    return frames



In [ ]:
def normalize(frame):
    center = np.mean(frame, axis=0)
    centered = frame - center

    dist = np.linalg.norm(centered, axis=1)
    rg = np.sqrt(np.mean(dist**2))

    if rg < 1e-6:
        return centered.astype(np.float32)

    return (centered / rg).astype(np.float32)

In [ ]:
import h5py
from tqdm import tqdm

try:
    frames = read_pdb_fixed_frames(FILE_PATH, atoms_per_frame=10000)
    print("Frames shape:", frames.shape)
    # Use atom count from first frame
    target_atoms = frames[0].shape[0]

    print(f"⚙️ Using {target_atoms} atoms")

    with h5py.File(OUTPUT_FILE, 'w') as f:
        dset = f.create_dataset(
            "coordinates",
            shape=(0, target_atoms, 3),
            maxshape=(None, target_atoms, 3),
            dtype="float32",
            chunks=(1, target_atoms, 3),
            compression="gzip"
        )

        for frame in tqdm(frames):
            try:
                if frame.shape[0] < target_atoms:
                    continue

                frame = frame[:target_atoms]

                norm = normalize(frame)

                dset.resize((dset.shape[0] + 1), axis=0)
                dset[-1] = norm

            except Exception as e:
                print("⚠️ Frame skipped:", e)

    print("\n✅ DONE — File processed successfully")

except Exception as e:
    print("\n❌ ERROR:", e)

Checking part (No need to run them)

In [ ]:
import h5py

with h5py.File(OUTPUT_FILE, 'r') as f:
    data = f["coordinates"]
    print("Shape:", data.shape)
    print("Sample range:", data[0].min(), data[0].max())

In [ ]:
import h5py

with h5py.File(OUTPUT_FILE, 'r') as f:
    d = f["coordinates"]
    print("Shape:", d.shape)
    print("Dtype:", d.dtype)